In [26]:
import glob
print(glob.glob("build/*"))

['build/Reconnection', 'build/Energy', 'build/CMakeFiles', 'build/Makefile', 'build/cmake_install.cmake', 'build/Polygon', 'build/Edge', 'build/Cell', 'build/CMakeCache.txt', 'build/Vertex', 'build/Run', 'build/tvm']


In [15]:
import glob
import pandas as pd
import numpy as np
dir = sorted(glob.glob("patternA/*.bulk.txt"))[-1]

# remove the part before the last slash in dir
file = dir.split("/")[-1]
print(dir)
dir = "patternA/"
df = pd.read_csv("{}initial_stress.csv".format(dir))
print(df)
cellID_to_stress = {int(i): float(stress) for i, stress in zip(df["cellID"].to_numpy(), df["Stress"].to_numpy())}
print(cellID_to_stress)
print(np.loadtxt("{}costs.txt".format(dir)))

patternA/0014.bulk.txt
   cellID    Stress
0     142  0.277201
{142: 0.2772011738735123}
[1.57e-04 9.56e-05 6.59e-05 4.53e-05 3.12e-05 2.16e-05 1.49e-05 1.04e-05
 7.20e-06 5.00e-06 3.48e-06 2.43e-06 1.68e-06 1.17e-06 8.21e-07]


In [ ]:
## create cross section of a periodic tissue
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob

dir = "7_bidisperse_5_4.9_0.5_increase/"
# dir = "7_mono_0.5_decrease/"
df = pd.read_csv("{}stresses.csv".format(dir))
print(df["CellID"])
training_cell = df["CellID"].to_numpy()[0]
ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
final_iter = max(ids)
file = "{}.bulk.txt".format(final_iter)
sample = PeriodicTissue.from_config(dir,file)
min = FIREminimization.periodic_tissue(sample)
min.load_cell_parameters("cellParameters.{}.input".format(final_iter))

for cellID, cell in sample.cells_.items():
    cell.vtk_scalar_ = cell.s0_
    cell.vtk_scalar_ = stress.calculate_max_shear_stress(sample, cellID)
normal = np.array([1,0,0])
center = sample.cells_[training_cell].center_
print(sample.cells_[training_cell].s0_)
makeSampleCrossSection(sample=min._config, center = center, normal = normal, filename = "cross_section.vtk")
single_cell = sample.extract_cell(training_cell)
makeSampleCrossSection(sample=single_cell, center=center, normal = normal, filename="single_cell_cross_section.vtk")



0    331
Name: CellID, dtype: int64
5.0


In [3]:
from toolbox.cSection import makeSampleCrossSection
from toolbox.patterns import Patterns
from toolbox.periodic import PeriodicTissue
from toolbox.minimization import FIREminimization
from toolbox import stress
import matplotlib.pyplot as plt
from toolbox import stress
import os
import numpy as np
import pandas as pd
import glob
dir = "8_0_5_1_sample/"
df = pd.read_csv("{}stresses.csv".format(dir))
print(df["CellID"])
# ids = [int(os.path.basename(f).split(".")[0]) for f in glob.glob("{}*.bulk.txt".format(dir))]
# ids = sorted(ids)
# final_iter = 33
# for iter in range(final_iter+1):
iter = 33
file = "{}.bulk.txt".format(iter)
sample = PeriodicTissue.from_config(dir,file)
min = FIREminimization.periodic_tissue(sample)
min.load_cell_parameters("cellParameters.{}.input".format(iter))
for i, row in df.iterrows():
    cellID = row["CellID"]
    target_stress = row["Target"]
    cell = sample.cells_[cellID]
    # shear = stress.calculate_max_shear_stress(sample, cellID)
    # cell.max_shear_stress_ = shear
    # vtk_scalar = abs(cell.max_shear_stress_ - target_stress)/ target_stress
    for polygonID in cell.polygons_:
        polygon = sample.polygons_[polygonID]
        # polygon.vtk_scalar_ = vtk_scalar
for cellID in df["CellID"].to_numpy():
    # sample.write_cell_collection_vtk(df["CellID"].to_numpy(),"{}.target_cells.vtk".format(iter),use_scalar=True)
    sample.write_cell_collection_vtk([cellID],"{}.single.vtk".format(cellID),use_scalar=False)



0     11
1    157
2    494
3    166
4    378
Name: CellID, dtype: int64
